# Experiment 4: Summarizing Academic Reports

CSL75 Skill Enhancement Laboratory.

A pre-trained Hugging Face model summarizes a long academic report. The report
is longer than the model's 1024 token window, so summarization runs as a map
reduce: summarize each chunk, then summarize the summaries.

The paper's own abstract is held out of the model input and used as a human
written reference, so the summary can be scored with ROUGE rather than judged
by eye. An extractive lead-3 baseline is scored the same way for comparison.

Before running: Runtime, Change runtime type, T4 GPU.

## 1. Check the runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader \
    || echo "No GPU detected. It will still run on CPU, just slower."

## 2. Install dependencies

In [ ]:
!pip install -q "transformers>=4.44" "accelerate>=0.33" "sentencepiece>=0.2" "pypdf>=4.0"

## 3. Get the code

Cloned straight from the repository, so there is nothing to upload. Run this
cell again after any edit: it pulls the latest commit and reloads the modules.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/rhuzaii/genaitut.git"
ROOT = Path("/content/genaitut")
PROJECT = ROOT / "experiment4_summarization"
MODULES = ("config", "document", "chunking", "evaluate", "summarize", "report", "run_summarizer")

if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--quiet", REPO, str(ROOT)], check=True)

if not PROJECT.exists():
    raise FileNotFoundError(f"{PROJECT} is missing. Check that the clone succeeded.")

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

for name in MODULES:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

print(f"working in {PROJECT}")
print("modules  :", sorted(f.name for f in PROJECT.glob("*.py")))

## 4. Inspect the document

The loader separates the abstract from the body. Confirm the split is clean
before summarizing, since a leaked abstract would invalidate the evaluation.

In [ ]:
import document

report_doc = document.load("documents/sample_report.txt")

print("title   :", report_doc.title)
print("stats   :", report_doc.stats())
print("abstract leaked into body:", report_doc.abstract[:40] in report_doc.body)
print()
print("ABSTRACT (held out, used as the reference)")
print(report_doc.abstract)

## 5. Run the summarizer

Downloads the model weights on first run, roughly 3 GB for both models, then
summarizes. Two to four minutes on a T4.

In [ ]:
import run_summarizer

run_summarizer.main([
    "--document", "documents/sample_report.txt",
    "--models", "bart", "distilbart",
    "--output", "outputs",
])

## 6. Read the summaries side by side

In [ ]:
import json
import textwrap
from pathlib import Path

manifest = json.loads((Path("outputs") / "manifest.json").read_text())

print("REFERENCE ABSTRACT")
print(textwrap.fill(manifest["reference"], 88))
print()

for item in manifest["summaries"].values():
    metrics = item["metrics"]
    print("=" * 88)
    print(f"{item['label'].upper()}  ({metrics['summary_words']} words, "
          f"{metrics['compression_ratio']}x compression)")
    print(textwrap.fill(item["summary"], 88))
    print()

In [ ]:
print(f"{'summary':<22}{'R-1':>9}{'R-2':>9}{'R-L':>9}{'words':>8}{'novel':>8}")
print("-" * 65)
for item in manifest["summaries"].values():
    m = item["metrics"]
    print(f"{item['label']:<22}"
          f"{m['rouge1']['f1']:>9.4f}{m['rouge2']['f1']:>9.4f}{m['rougeL']['f1']:>9.4f}"
          f"{m['summary_words']:>8}{m['novel_bigram_ratio']:>8.2f}")
print()
print("Novel bigram ratio near zero means the model copied rather than rewrote.")

## 7. Optional: summarize your own paper

Upload a PDF, then point the summarizer at it. Works best on a paper that has
a heading literally reading "Abstract", since that is how the reference is
located. Without one, ROUGE is skipped and only the compression and novelty
figures are reported.

In [ ]:
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
Path("documents").mkdir(exist_ok=True)
Path("documents", name).write_bytes(uploaded[name])

run_summarizer.main([
    "--document", f"documents/{name}",
    "--models", "bart",
    "--output", "outputs_custom",
])

## 8. Collect the outputs

`outputs/report.md` has the settings, chunk table, every summary and the metric
table already filled in. The observation prompts are left blank on purpose.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("experiment4_outputs", "zip", "outputs")
files.download("experiment4_outputs.zip")